In [11]:
import pandas as pd
import glob
import re

# Load all CSV files in the current directory
csv_files = glob.glob('*.csv')  # This will match all CSV files in the current folder

# List to store DataFrames
csv_data_frames = []

# Regular expression to match participant IDs
pattern = re.compile(r'^(\d+)(?:-(\d+))?\.csv')  # Matches "6.csv", "7.csv", "10-1.csv", etc.

# Loop through each CSV file and read it into a DataFrame
for csv_file in csv_files:
    # Check if the filename matches the expected format
    match = pattern.match(csv_file)
    
    if match:
        # Extract participant ID
        participant_id = int(match.group(1))  # Get the ID part
        # Read the CSV file into a DataFrame
        df = pd.read_csv(csv_file)
        df['Participant_ID'] = participant_id  # Add a column for Participant ID
        csv_data_frames.append(df)
    else:
        print(f"Skipping file with invalid naming format: {csv_file}")

# Combine all CSV DataFrames into a single DataFrame
if csv_data_frames:  # Check to ensure there are DataFrames to combine
    combined_csv_data = pd.concat(csv_data_frames, ignore_index=True)
    print("Combined Data:")
    print(combined_csv_data.head())  # Show the first few rows of the combined data
else:
    print("No valid CSV data found. Make sure the CSV files are in the correct format.")

Skipping file with invalid naming format: comparison_actual_vs_predicted.csv
Combined Data:
            Time  Zone_0  Zone_1  Zone_2  Zone_3  Zone_4  Zone_5  Zone_6  \
0  1747995538000    3357     243    3708     314    2057    2079     289   
1  1747995538200     273     295     315     340    2073    2095     289   
2  1747995538400     317     283     318     340    2070    2095     289   
3  1747995538600     295     290     315     340    2073    2095     289   
4  1747995538800     317     283     318     340    2070    2085     289   

   Zone_7  Zone_8  ...  Zone_57  Zone_58  Zone_59  Zone_60  Zone_61  Zone_62  \
0     327     248  ...      217      208      210      212      239      210   
1     327     276  ...      217      208      212      215      246      198   
2     327     248  ...      217      208      210      215      246      198   
3    1447     276  ...      221      208      210      215      246      197   
4    1447     248  ...      221      208      210  

In [ ]:
hydrocare_data = pd.read_excel('Hydrocare_ffinal.xlsx')  # Use the Hydrocare data file

# Print column names for confirmation
print("Hydrocare Columns:", hydrocare_data.columns)

# Create a dictionary from the Hydrocare data with ID as the key
# Use the exact names found in the printed output
hydrocare_dict = hydrocare_data.set_index('ID')[['containers weight', 'Density']].to_dict(orient='index')

# Initialize lists to store actual volumes
actual_volumes = []


# Iterate through the combined CSV data and calculate actual volume
for index, row in combined_csv_data.iterrows():
    participant_id = row['Participant_ID']  # Get the participant ID
    
    # Check if the participant ID exists in the hydrocare dictionary
    if participant_id in hydrocare_dict:
        container_weight = hydrocare_dict[participant_id]['containers weight']  # Use correct name
        density = hydrocare_dict[participant_id]['Density']
        
        # Get the total weight (whole weight) from the combined data
        total_weight = row['Volume']  # Replace 'volume' with the actual column name for weight

        # Calculate actual volume: (total weight - container weight) / density
        if density != 0:  # To avoid division by zero
            net_weight = total_weight - container_weight
            actual_volume = net_weight / density
            actual_volumes.append(actual_volume)
            
        else:
            actual_volumes.append(None)
            
    else:
        actual_volumes.append(None)
        

# Add the calculated volumes as new columns to the combined CSV data
combined_csv_data['Actual_Volume'] = actual_volumes

# Display the first few rows of the updated DataFrame for verification
print(combined_csv_data.head())  # Show relevant columns

# Optional: Export the updated DataFrame to a new Excel file
output_file_path = 'updated_combined_data_with_actual_volume.xlsx'  # Specify the output file name
combined_csv_data.to_excel(output_file_path, index=False)  # Export to Excel
print(f"Updated data with actual volume exported to {output_file_path}")

Hydrocare Columns: Index([                           'ID',                        'Gender',
                   'containers weight',                       'Density',
                        'whole weight',                 'type of drink',
                                'mate',                         'straw',
       'weight at first after opening',               'after first sip',
                    'after second sip',                           '3rd',
                                 '4th',                               5,
                                     6,                               7,
                                     8,                               9,
                                    10,                              11,
                                    12,                              13,
                                    14,                   'Unnamed: 25',
                         'Unnamed: 26',                   'Unnamed: 27',
                         'Unname

In [ ]:
import pandas as pd

# Load the Hydrocare data file
hydrocare_data = pd.read_excel('Hydrocare_ffinal.xlsx')  # Use the Hydrocare data file

# Print column names for confirmation
print("Hydrocare Columns:", hydrocare_data.columns)

# Create a dictionary from the Hydrocare data with ID as the key
hydrocare_dict = hydrocare_data.set_index('ID')[['containers weight', 'Density', 'straw', 'Gender']].to_dict(orient='index')

# Initialize lists to store actual volumes and assign straw usage
actual_volumes = []

# Iterate through the combined CSV data and calculate actual volume
for index, row in combined_csv_data.iterrows():
    participant_id = row['Participant_ID']  # Get the participant ID
    
    # Check if the participant ID exists in the hydrocare dictionary
    if participant_id in hydrocare_dict:
        container_weight = hydrocare_dict[participant_id]['containers weight']  # Use correct name
        density = hydrocare_dict[participant_id]['Density']
        straw_usage = hydrocare_dict[participant_id]['straw']  # Get straw information
        gender = hydrocare_dict[participant_id]['Gender']  # Get gender information
        
        # Get the total weight (whole weight) from the combined data
        total_weight = row['Volume']  # Replace 'Volume' with the actual column name for weight

        # Calculate actual volume: (total weight - container weight) / density
        if density != 0:  # To avoid division by zero
            net_weight = total_weight - container_weight
            actual_volume = net_weight / density
            actual_volumes.append((actual_volume, straw_usage, gender, container_weight))  # Append volume, straw usage, and gender
        else:
            actual_volumes.append((None, straw_usage, gender, container_weight))
    else:
        actual_volumes.append((None, None, None, None))  # If ID is not found, append None for all

# Add the calculated volumes and additional information to the combined CSV data
combined_csv_data['Actual_Volume'] = [volume[0] for volume in actual_volumes]
combined_csv_data['Straw'] = [volume[1] for volume in actual_volumes]
combined_csv_data['Gender'] = [volume[2] for volume in actual_volumes]
combined_csv_data['Container_Weight'] = [volume[3] for volume in actual_volumes]

# # Create a sip_id column to count drinking labels
# sip_ids = []
# previous_state = None  # None, "drinking", or "not drinking"
# current_sip_id = 0
# for index, row in combined_csv_data.iterrows():
#     actual_volume = row['Actual_Volume']
    
#     # Determine current state based on actual_volume
#     if actual_volume is not None and actual_volume > 0:
#         current_state = "drinking"  # The participant is drinking
#     else:
#         current_state = "not drinking"  # The participant is not drinking

#     # Increment sip_id only on transition
#     if current_state != previous_state:
#         current_sip_id += 1  # Increment sip ID on change of state
#         previous_state = current_state  # Update previous state

#     sip_ids.append(current_sip_id)  # Append the current sip_id

# Assign calculated sip_id values to the DataFrame
combined_csv_data['sip_id'] = sip_ids
# Split the data into two separate DataFrames based on straw usage
with_straw = combined_csv_data[combined_csv_data['Straw'] == 'Y']
without_straw = combined_csv_data[combined_csv_data['Straw'] == 'N']

# Display the first few rows of each DataFrame for verification
print("Data with Straw:\n", with_straw.head())
print("Data without Straw:\n", without_straw.head())

# Optional: Export the two separate DataFrames to new Excel files
with_straw_file_path = 'data_with_straw.xlsx'  # Specify the output file name for straw users
without_straw_file_path = 'data_without_straw.xlsx'  # Specify the output file name for non-straw users

with_straw.to_excel(with_straw_file_path, index=False)  # Export to Excel
without_straw.to_excel(without_straw_file_path, index=False)  # Export to Excel

print(f"Data with straw exported to {with_straw_file_path}")
print(f"Data without straw exported to {without_straw_file_path}")